# VIVO Backgammon — Entrenamiento red ancha en GPU (Colab)

**Antes de ejecutar:**
1. Has hecho `git push -u origin 260815-wide-net-gpu` del repo local.
2. En **Cell 3** reemplazas `REPO_URL` por la URL real de tu repo en GitHub.
3. `Runtime` → `Change runtime type` → `GPU` (T4/A100).
4. Ejecuta las celdas en orden. La de entrenamiento corre *para siempre*; mírala hasta que dos `eval` consecutivos den `rate >= 0.60`, luego `Runtime` → `Interrupt execution`.
5. Descarga `model_weights.json` (última celda) y colócalo en `public/` del proyecto local.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar Node 22

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y nodejs
!node --version && npm --version

## 3. Clonar la rama de trabajo

**REEMPLAZA** `REPO_URL` por tu URL real (ej. `https://github.com/tu_usuario/BACKGAMMON-VIVO.git`).

In [ ]:
REPO_URL = "https://github.com/TU_USUARIO/BACKGAMMON-VIVO.git"  # <-- CAMBIA ESTO
BRANCH = "260815-wide-net-gpu"

!rm -rf vivo_train_repo
!git clone --branch {BRANCH} {REPO_URL} vivo_train_repo
%cd vivo_train_repo

## 4. Instalar dependencias (tfjs-node-gpu enlaza con CUDA de Colab)

In [ ]:
!npm ci

## 5. Entrenar (self-play on-policy, vs heurística)

Corre indefinidamente. Observa las líneas `{"event":"eval",...,"rate":X}`.
**Para cuando dos `eval` consecutivos den `rate >= 0.60`** → `Runtime` → `Interrupt execution`.

Si plateau < 0.60 tras ~5000 partidas: detén, crea un commit cambiando `hidden` a `[512,256,128]` en `src/features/ai-worker/training/net-arch.ts`, haz push, y reinicia este notebook desde la celda 3.

In [ ]:
!npx tsx src/features/ai-worker/training/cli.ts \
  --games=100000 --opponent=heuristic --label=outcome \
  --exploration=0.15 --max-moves=400 --epochs=3 \
  --eval-every=250 --eval-games=200 --nn-blend=1 --save-every=50

## 6. Descargar pesos entrenados

Ejecuta SOLO tras interrumpir la celda 5 (los pesos se guardan cada `--save-every=50` partidas).

In [ ]:
from google.colab import files
files.download('public/model_weights.json')

## 7. Verificar forma del peso (opcional, local)

En tu PC, tras colocar el archivo en `public/model_weights.json`:
```bash
npx tsx src/features/ai-worker/training/tournament.ts   # debe dar rate >= 0.60 (PASS)
npx vite build                                          # build OK
```
La capa 0 debe tener `shape:[198,256]` (no `[198,40]`).